#### steps taken:
1. Read silver `constructors` table
2. Read gold `ref_nationality_region` table
3. Join the data from `constructors` with `ref_nationality_region` using `nationality`
4. Select the required columns
   - constructors.constructor_id
   - constructors.constructor_name
   - constructors.nationality
   - ref_nationality_region.region
5. Write the transformed data to gold `dim_constructors` table

In [0]:
%run ../environment_config

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalog}.{gold_schema}.dim_constructors"

In [0]:
constructor_df = spark.table(f"{catalog}.{silver_schema}.constructors")
ref_nationality_region = spark.table(f"{catalog}.{gold_schema}.ref_nationality_region")

In [0]:
dim_constructor_df =(
    constructor_df.alias("c")
                  .join(
                         ref_nationality_region.alias("n"),
                         F.col("c.nationality") == F.col("n.nationality"),
                         "left"
                        )
                  .select(
                     constructor_df.constructor_id,
                     constructor_df.constructor_name,
                     constructor_df.nationality,
                     ref_nationality_region.region.alias("nationality_region")
                  )
                   
)

In [0]:
display(dim_constructor_df)

In [0]:
(
    dim_constructor_df.write
                      .format("delta")
                      .mode("overwrite")
                      .saveAsTable(target_table)
)

In [0]:
display(spark.table(target_table))